## Models
| Model | Library | Notes |
|-------|---------|-------|
| xRFM | `xrfm` (official) | GPU-accelerated via CUDA 12 |
| XGBoost | `xgboost` | GPU-enabled (`tree_method='hist'`, `device='cuda'`) |
| Random Forest | `sklearn` | CPU-only baseline |

Hyperparameters are tuned on validation set (via xRFM's built-in tuning for xRFM; small grid search for XGBoost; defaults + sensible configs for RF). Final metrics are reported on the held-out test set only

In [1]:

import numpy as np
import pandas as pd
import os
import pickle
import time
import warnings
warnings.filterwarnings('ignore')


import torch
from xrfm import xRFM
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier


from sklearn.metrics import (
    mean_squared_error, accuracy_score, roc_auc_score,
    root_mean_squared_error
)


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


DATASET_FILES = [
    'concrete',
    'energy',
    'bike_sharing',
    'online_shoppers',
    'shuttle',
]

datasets = {}
for name in DATASET_FILES:
    with open(f'data/processed/{name}.pkl', 'rb') as f:
        datasets[name] = pickle.load(f)
    d = datasets[name]
    print(f"  loaded {name}: n_train={d['n_train']}, d={d['n_features']}, task={d['task_type']}")

print(f"\n{len(datasets)} datasets loaded.")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
  loaded concrete: n_train=659, d=8, task=regression
  loaded energy: n_train=491, d=8, task=regression
  loaded bike_sharing: n_train=11122, d=61, task=regression
  loaded online_shoppers: n_train=7891, d=29, task=classification
  loaded shuttle: n_train=37120, d=7, task=classification

5 datasets loaded.


In [2]:
def evaluate_regression(y_true, y_pred):
    """Compute RMSE for regression."""
    rmse = root_mean_squared_error(y_true, y_pred)
    return {'RMSE': rmse}


def evaluate_classification(y_true, y_pred_labels, y_pred_proba, n_classes):
    """Compute Accuracy and AUC-ROC for classification.
    Robust to cases where predicted proba matrix doesn't cover all classes."""
    acc = accuracy_score(y_true, y_pred_labels)
    labels = np.arange(n_classes)

    if n_classes == 2:
        if y_pred_proba.ndim == 2:
            proba_pos = y_pred_proba[:, 1]
        else:
            proba_pos = y_pred_proba
        auc = roc_auc_score(y_true, proba_pos)
    else:
        if y_pred_proba.shape[1] != n_classes:
            fixed = np.zeros((len(y_pred_proba), n_classes))
            fixed[:, :y_pred_proba.shape[1]] = y_pred_proba
            y_pred_proba = fixed / (fixed.sum(axis=1, keepdims=True) + 1e-12)
        auc = roc_auc_score(y_true, y_pred_proba, multi_class='ovr',
                            average='macro', labels=labels)
    return {'Accuracy': acc, 'AUC-ROC': auc}


def time_inference_per_sample(predict_fn, X_test, n_reps=3):
    """Measure average inference time per sample over n_reps runs."""
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        _ = predict_fn(X_test)
        t1 = time.perf_counter()
        times.append((t1 - t0) / len(X_test))
    return float(np.mean(times))


all_results = []

print("Evaluation helpers defined.")

Evaluation helpers defined.


In [3]:
def train_xrfm(data, dataset_name):
    """
    Train xRFM on a preprocessed dataset.

    Uses more conservative hyperparameters that avoid the 'best_alphas is None' bug.
    """
    print(f"\n{'='*60}")
    print(f"Training xRFM on {dataset_name}")
    print(f"{'='*60}")

    task = data['task_type']

    X_train = torch.tensor(data['X_train'], dtype=torch.float32, device=DEVICE)
    X_val   = torch.tensor(data['X_val'],   dtype=torch.float32, device=DEVICE)
    X_test  = torch.tensor(data['X_test'],  dtype=torch.float32, device=DEVICE)

    if task == 'classification':
        n_classes = data['n_classes']
        y_train_oh = np.zeros((len(data['y_train']), n_classes), dtype=np.float32)
        y_train_oh[np.arange(len(data['y_train'])), data['y_train']] = 1.0
        y_val_oh = np.zeros((len(data['y_val']), n_classes), dtype=np.float32)
        y_val_oh[np.arange(len(data['y_val'])), data['y_val']] = 1.0
        y_train = torch.tensor(y_train_oh, dtype=torch.float32, device=DEVICE)
        y_val   = torch.tensor(y_val_oh,   dtype=torch.float32, device=DEVICE)
        tuning_metric = 'accuracy' if n_classes > 2 else 'auc'
    else:
        y_train = torch.tensor(data['y_train'].reshape(-1, 1), dtype=torch.float32, device=DEVICE)
        y_val   = torch.tensor(data['y_val'].reshape(-1, 1),   dtype=torch.float32, device=DEVICE)
        tuning_metric = 'mse'

  
    rfm_params = {
        'model': {
            'kernel': 'l2',
            'bandwidth': 10.0,
            'exponent': 1.0,
            'diag': False,
            'bandwidth_mode': 'adaptive',   
        },
        'fit': {
            'reg': 1e-2,                    
            'iters': 10,
            'early_stop_rfm': False,
            'verbose': False,
        },
    }

    
    n_train = len(X_train)
    min_subset = max(4000, n_train // 4)

    model = xRFM(
        rfm_params=rfm_params,
        device=DEVICE,
        tuning_metric=tuning_metric,
        min_subset_size=min_subset,
    )

    t0 = time.perf_counter()
    model.fit(X_train, y_train, X_val, y_val)
    train_time = time.perf_counter() - t0
    print(f"  Training time: {train_time:.2f}s (min_subset_size={min_subset})")

    def predict_fn(X):
        with torch.no_grad():
            return model.predict(X)

    y_pred_raw = predict_fn(X_test)
    if isinstance(y_pred_raw, torch.Tensor):
        y_pred_raw = y_pred_raw.cpu().numpy()

    if task == 'regression':
        y_pred = y_pred_raw.ravel()
        metrics = evaluate_regression(data['y_test'], y_pred)
    else:
        if y_pred_raw.ndim == 1:
            proba = np.column_stack([1 - y_pred_raw, y_pred_raw])
            y_pred_labels = (y_pred_raw > 0.5).astype(int)
        elif y_pred_raw.shape[1] == 1:
            scores = y_pred_raw.ravel()
            proba = np.column_stack([1 - scores, scores])
            y_pred_labels = (scores > 0.5).astype(int)
        else:
            scores = y_pred_raw
            exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
            proba = exp_scores / exp_scores.sum(axis=1, keepdims=True)
            y_pred_labels = proba.argmax(axis=1)
        metrics = evaluate_classification(
            data['y_test'], y_pred_labels, proba, data['n_classes']
        )

    inf_time_per_sample = time_inference_per_sample(predict_fn, X_test, n_reps=3)

    result = {
        'dataset': dataset_name,
        'model': 'xRFM',
        'task': task,
        **metrics,
        'train_time_s': train_time,
        'inference_time_per_sample_s': inf_time_per_sample,
        'n_train': data['n_train'],
        'n_features': data['n_features'],
    }

    metric_str = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"  Metrics: {metric_str}")
    print(f"  Inference time/sample: {inf_time_per_sample*1000:.4f} ms")

    return result, model

print("xRFM training function defined (v4 — stable config).")

xRFM training function defined (v4 — stable config).


In [4]:

all_results = []
xrfm_models = {}

for name in DATASET_FILES:
    try:
        result, model = train_xrfm(datasets[name], name)
        all_results.append(result)
        xrfm_models[name] = model
    except Exception as e:
        print(f"\n⚠️  xRFM FAILED on {name}: {type(e).__name__}: {e}")
        print(f"   Skipping this dataset for xRFM; continuing with others.\n")


os.makedirs('data/models', exist_ok=True)
for name, model in xrfm_models.items():
    with open(f'data/models/xrfm_{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"  saved xrfm_{name}.pkl")

print(f"\n✓ xRFM trained on {len(xrfm_models)}/{len(DATASET_FILES)} datasets.")
print(f"  Results collected so far: {len(all_results)}")


Training xRFM on concrete
None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|                                                                            | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 659, d: 8, and nval: 165
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 0: 0.41336679458618164 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 1: 0.011843442916870117 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 2: 0.014060020446777344 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 3: 0.00999760627746582 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 4: 0.013493061065673828 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 5: 0.008435726165771484 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659


Building trees:   0%|                                                                            | 0/1 [00:00<?, ?it/s]


Using SVD
Time taken for round 6: 0.01652383804321289 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 7: 0.016611337661743164 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 8: 0.013216495513916016 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 9: 0.01696944236755371 seconds
Resetting adaptive bandwidth
Tree has no split, stopping training
  Training time: 0.59s (min_subset_size=4000)
Using hard routing for tree prediction
Using hard routing for tree prediction
Using hard routing for tree prediction
Using hard routing for tree prediction
  Metrics: RMSE=5.7016
  Inference time/sample: 0.0089 ms

Training xRFM on energy
None
Fitting xRFM with 1 trees and 0 iterations per tree


...

Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([19648, 7]) y_train torch.Size([19648, 7]) X_val torch.Size([1035, 7]) y_val torch.Size([1035, 7])
Fitting RFM with ntrain: 19648, d: 7, and nval: 1035
Using cheap batch size
Optimal M batch size: 1711
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([19646, 7]) y_train torch.Size([19646, 7]) X_val torch.Size([1034, 7]) y_val torch.Size([1034, 7])
Fitting RFM with ntrain: 19646, d: 7, and nval: 1034
Using cheap batch size
Optimal M batch size: 1710
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([12694, 7]) y_train torch.Size([12694, 7]) X_val torch.Size([669, 7]) y_val torch.Size([669, 7])
Fitting RFM with ntrain: 12694, d: 7, and nval: 669
Using cheap batch size
Optimal M batch size: 1710
Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([7617, 7]) y_train torch.Size([7617, 7]) X_val to

Building trees: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:22<00:00, 22.80s/it]


Using SVD
Time taken for round 5: 0.04627490043640137 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 1708
Using SVD
Time taken for round 6: 0.0365140438079834 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 1708
Using SVD
Time taken for round 7: 0.023672103881835938 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 1708
Using SVD
Time taken for round 8: 0.03177452087402344 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 1708
Using SVD
Time taken for round 9: 0.03191709518432617 seconds
Resetting adaptive bandwidth


Tuning split temperature: 100%|████████████████████████████████████████████████████████| 36/36 [00:02<00:00, 13.38it/s]


Selected split_temperature=0.0 based on validation accuracy=0.998276
  Training time: 25.53s (min_subset_size=9280)
Using hard routing for tree prediction
Using hard routing for tree prediction
Using hard routing for tree prediction
Using hard routing for tree prediction
  Metrics: Accuracy=0.7867, AUC-ROC=0.6121
  Inference time/sample: 0.0058 ms
  saved xrfm_concrete.pkl
  saved xrfm_energy.pkl
  saved xrfm_bike_sharing.pkl
  saved xrfm_online_shoppers.pkl
  saved xrfm_shuttle.pkl

✓ xRFM trained on 5/5 datasets.
  Results collected so far: 5


In [5]:
print("xRFM models in cache:", list(xrfm_models.keys()))
print("\nxRFM results in all_results:")
for r in all_results:
    if r['model'] == 'xRFM':
        metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in r.items() if k in ['RMSE', 'Accuracy', 'AUC-ROC'])
        print(f"  {r['dataset']}: {metrics_str}")
        

xRFM models in cache: ['concrete', 'energy', 'bike_sharing', 'online_shoppers', 'shuttle']

xRFM results in all_results:
  concrete: RMSE=5.7016
  energy: RMSE=0.6234
  bike_sharing: RMSE=41.7977
  online_shoppers: Accuracy=0.8909, AUC-ROC=0.7334
  shuttle: Accuracy=0.7867, AUC-ROC=0.6121


In [6]:
def train_xgboost(data, dataset_name):
    """
    Train XGBoost with light validation-based tuning.

    Uses GPU acceleration via `device='cuda'`, hist-based tree method.
    Small grid search over tree depth and learning rate using the validation set.
    """
    print(f"\n{'='*60}")
    print(f"Training XGBoost on {dataset_name}")
    print(f"{'='*60}")

    task = data['task_type']
    X_train = data['X_train']
    X_val   = data['X_val']
    X_test  = data['X_test']
    y_train = data['y_train']
    y_val   = data['y_val']
    y_test  = data['y_test']

    # Define a small hyperparameter grid
    depth_grid = [4, 6, 8]
    lr_grid = [0.05, 0.1]
    n_estimators_cap = 500

    # Common kwargs
    base_kwargs = dict(
        tree_method='hist',
        device='cuda' if torch.cuda.is_available() else 'cpu',
        n_estimators=n_estimators_cap,
        random_state=RANDOM_SEED,
        verbosity=0,
        early_stopping_rounds=20,
    )

    if task == 'regression':
        Model = xgb.XGBRegressor
        base_kwargs['objective'] = 'reg:squarederror'
        score_higher_better = False  # RMSE
    else:
        n_classes = data['n_classes']
        Model = xgb.XGBClassifier
        if n_classes == 2:
            base_kwargs['objective'] = 'binary:logistic'
            base_kwargs['eval_metric'] = 'auc'
        else:
            base_kwargs['objective'] = 'multi:softprob'
            base_kwargs['eval_metric'] = 'mlogloss'
            base_kwargs['num_class'] = n_classes
        score_higher_better = True 

    
    t0 = time.perf_counter()

    best_score = None
    best_params = None
    best_model = None

    for depth in depth_grid:
        for lr in lr_grid:
            kw = dict(base_kwargs, max_depth=depth, learning_rate=lr)
            m = Model(**kw)
            m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

           
            if task == 'regression':
                val_pred = m.predict(X_val)
                from sklearn.metrics import root_mean_squared_error
                score = root_mean_squared_error(y_val, val_pred)
                is_better = (best_score is None) or (score < best_score)
            else:
                if n_classes == 2:
                    val_proba = m.predict_proba(X_val)[:, 1]
                    score = roc_auc_score(y_val, val_proba)
                else:
                    val_pred = m.predict(X_val)
                    score = accuracy_score(y_val, val_pred)
                is_better = (best_score is None) or (score > best_score)

            if is_better:
                best_score = score
                best_params = {'max_depth': depth, 'learning_rate': lr}
                best_model = m

    train_time = time.perf_counter() - t0
    print(f"  Training time (incl. tuning): {train_time:.2f}s")
    print(f"  Best params: {best_params} | val score: {best_score:.4f}")

   
    def predict_fn(X):
        return best_model.predict(X)

    if task == 'regression':
        y_pred = best_model.predict(X_test)
        metrics = evaluate_regression(y_test, y_pred)
    else:
        y_pred_labels = best_model.predict(X_test)
        y_pred_proba = best_model.predict_proba(X_test)
        metrics = evaluate_classification(y_test, y_pred_labels, y_pred_proba, n_classes)

    inf_time_per_sample = time_inference_per_sample(predict_fn, X_test, n_reps=3)

    result = {
        'dataset': dataset_name,
        'model': 'XGBoost',
        'task': task,
        **metrics,
        'train_time_s': train_time,
        'inference_time_per_sample_s': inf_time_per_sample,
        'n_train': data['n_train'],
        'n_features': data['n_features'],
    }

    metric_str = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"  Test metrics: {metric_str}")
    print(f"  Inference time/sample: {inf_time_per_sample*1000:.4f} ms")
    return result, best_model

print("XGBoost training function defined.")

XGBoost training function defined.


In [7]:

xgb_models = {}

for name in DATASET_FILES:
    try:
        result, model = train_xgboost(datasets[name], name)
       
        all_results = [r for r in all_results if not (r['dataset']==name and r['model']=='XGBoost')]
        all_results.append(result)
        xgb_models[name] = model
    except Exception as e:
        print(f"\n⚠️  XGBoost FAILED on {name}: {type(e).__name__}: {e}")


for name, model in xgb_models.items():
    with open(f'data/models/xgb_{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"  saved xgb_{name}.pkl")

print(f"\n✓ XGBoost coverage: {len(xgb_models)}/{len(DATASET_FILES)}")
print(f"  Total results in registry: {len(all_results)}")


Training XGBoost on concrete
  Training time (incl. tuning): 4.72s
  Best params: {'max_depth': 6, 'learning_rate': 0.1} | val score: 4.8154
  Test metrics: RMSE=5.1542
  Inference time/sample: 0.0090 ms

Training XGBoost on energy
  Training time (incl. tuning): 9.82s
  Best params: {'max_depth': 4, 'learning_rate': 0.1} | val score: 0.4065
  Test metrics: RMSE=0.3542
  Inference time/sample: 0.0288 ms

Training XGBoost on bike_sharing
  Training time (incl. tuning): 19.00s
  Best params: {'max_depth': 6, 'learning_rate': 0.1} | val score: 43.9943
  Test metrics: RMSE=44.6793
  Inference time/sample: 0.0026 ms

Training XGBoost on online_shoppers
  Training time (incl. tuning): 3.17s
  Best params: {'max_depth': 6, 'learning_rate': 0.05} | val score: 0.9368
  Test metrics: Accuracy=0.9023, AUC-ROC=0.9291
  Inference time/sample: 0.0011 ms

Training XGBoost on shuttle
  Training time (incl. tuning): 50.86s
  Best params: {'max_depth': 4, 'learning_rate': 0.05} | val score: 0.9988
  Te

In [8]:
def train_random_forest(data, dataset_name):
    """
    Train Random Forest with light validation-based tuning over n_estimators and max_depth.
    Uses all CPU cores via n_jobs=-1.
    """
    print(f"\n{'='*60}")
    print(f"Training Random Forest on {dataset_name}")
    print(f"{'='*60}")

    task = data['task_type']
    X_train = data['X_train']
    X_val   = data['X_val']
    X_test  = data['X_test']
    y_train = data['y_train']
    y_val   = data['y_val']
    y_test  = data['y_test']

    n_est_grid = [100, 300]
    max_depth_grid = [None, 20]

    Model = RandomForestRegressor if task == 'regression' else RandomForestClassifier

    t0 = time.perf_counter()

    best_score = None
    best_params = None
    best_model = None

    for n_est in n_est_grid:
        for md in max_depth_grid:
            m = Model(
                n_estimators=n_est,
                max_depth=md,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            )
            m.fit(X_train, y_train)

            if task == 'regression':
                val_pred = m.predict(X_val)
                score = root_mean_squared_error(y_val, val_pred)
                is_better = (best_score is None) or (score < best_score)
            else:
                n_classes = data['n_classes']
                if n_classes == 2:
                    val_proba = m.predict_proba(X_val)[:, 1]
                    score = roc_auc_score(y_val, val_proba)
                else:
                    val_pred = m.predict(X_val)
                    score = accuracy_score(y_val, val_pred)
                is_better = (best_score is None) or (score > best_score)

            if is_better:
                best_score = score
                best_params = {'n_estimators': n_est, 'max_depth': md}
                best_model = m

    train_time = time.perf_counter() - t0
    print(f"  Training time (incl. tuning): {train_time:.2f}s")
    print(f"  Best params: {best_params} | val score: {best_score:.4f}")

    def predict_fn(X):
        return best_model.predict(X)

    if task == 'regression':
        y_pred = best_model.predict(X_test)
        metrics = evaluate_regression(y_test, y_pred)
    else:
        n_classes = data['n_classes']
        y_pred_labels = best_model.predict(X_test)
        y_pred_proba = best_model.predict_proba(X_test)
        metrics = evaluate_classification(y_test, y_pred_labels, y_pred_proba, n_classes)

    inf_time_per_sample = time_inference_per_sample(predict_fn, X_test, n_reps=3)

    result = {
        'dataset': dataset_name,
        'model': 'RandomForest',
        'task': task,
        **metrics,
        'train_time_s': train_time,
        'inference_time_per_sample_s': inf_time_per_sample,
        'n_train': data['n_train'],
        'n_features': data['n_features'],
    }

    metric_str = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"  Test metrics: {metric_str}")
    print(f"  Inference time/sample: {inf_time_per_sample*1000:.4f} ms")
    return result, best_model

print("Random Forest training function defined.")

Random Forest training function defined.


In [9]:
rf_models = {}

for name in DATASET_FILES:
    try:
        result, model = train_random_forest(datasets[name], name)
        all_results = [r for r in all_results if not (r['dataset']==name and r['model']=='RandomForest')]
        all_results.append(result)
        rf_models[name] = model
    except Exception as e:
        print(f"\n⚠️  Random Forest FAILED on {name}: {type(e).__name__}: {e}")

for name, model in rf_models.items():
    with open(f'data/models/rf_{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"  saved rf_{name}.pkl")

print(f"\n✓ Random Forest coverage: {len(rf_models)}/{len(DATASET_FILES)}")
print(f"  Total results in registry: {len(all_results)}")


Training Random Forest on concrete
  Training time (incl. tuning): 1.73s
  Best params: {'n_estimators': 300, 'max_depth': 20} | val score: 4.9152
  Test metrics: RMSE=6.0390
  Inference time/sample: 0.5053 ms

Training Random Forest on energy
  Training time (incl. tuning): 1.66s
  Best params: {'n_estimators': 100, 'max_depth': None} | val score: 0.5463
  Test metrics: RMSE=0.5352
  Inference time/sample: 0.3163 ms

Training Random Forest on bike_sharing
  Training time (incl. tuning): 7.52s
  Best params: {'n_estimators': 300, 'max_depth': None} | val score: 51.4690
  Test metrics: RMSE=49.8290
  Inference time/sample: 0.0562 ms

Training Random Forest on online_shoppers
  Training time (incl. tuning): 2.66s
  Best params: {'n_estimators': 300, 'max_depth': None} | val score: 0.9336
  Test metrics: Accuracy=0.9019, AUC-ROC=0.9184
  Inference time/sample: 0.0552 ms

Training Random Forest on shuttle
  Training time (incl. tuning): 4.10s
  Best params: {'n_estimators': 100, 'max_dept

In [10]:
results_df = pd.DataFrame(all_results)

results_df = results_df[[
    'dataset', 'model', 'task',
    'RMSE', 'Accuracy', 'AUC-ROC',
    'train_time_s', 'inference_time_per_sample_s',
    'n_train', 'n_features'
]]

dataset_order = DATASET_FILES
model_order = ['xRFM', 'XGBoost', 'RandomForest']
results_df['dataset'] = pd.Categorical(results_df['dataset'], categories=dataset_order, ordered=True)
results_df['model'] = pd.Categorical(results_df['model'], categories=model_order, ordered=True)
results_df = results_df.sort_values(['dataset', 'model']).reset_index(drop=True)

results_df.to_csv('results/tables/model_results.csv', index=False)
print(f"Saved: results/tables/model_results.csv ({len(results_df)} rows)")
print()
print(results_df.to_string(index=False))


Saved: results/tables/model_results.csv (15 rows)

        dataset        model           task      RMSE  Accuracy  AUC-ROC  train_time_s  inference_time_per_sample_s  n_train  n_features
       concrete         xRFM     regression  5.701559       NaN      NaN      0.590668                     0.000009      659           8
       concrete      XGBoost     regression  5.154179       NaN      NaN      4.717511                     0.000009      659           8
       concrete RandomForest     regression  6.039030       NaN      NaN      1.730973                     0.000505      659           8
         energy         xRFM     regression  0.623436       NaN      NaN      0.171693                     0.000013      491           8
         energy      XGBoost     regression  0.354208       NaN      NaN      9.824969                     0.000029      491           8
         energy RandomForest     regression  0.535221       NaN      NaN      1.663631                     0.000316      491   

In [11]:
def build_pivot(results_df, metric):
    sub = results_df[['dataset', 'model', metric]].copy()
    pivot = sub.pivot(index='dataset', columns='model', values=metric)
    pivot = pivot.reindex(index=dataset_order, columns=model_order)
    return pivot

print("=" * 70)
print("RMSE (regression, lower is better)")
print("=" * 70)
rmse_pivot = build_pivot(results_df[results_df['task']=='regression'], 'RMSE')
print(rmse_pivot.round(4).to_string())

print("\n" + "=" * 70)
print("Accuracy (classification, higher is better)")
print("=" * 70)
acc_pivot = build_pivot(results_df[results_df['task']=='classification'], 'Accuracy')
print(acc_pivot.round(4).to_string())

print("\n" + "=" * 70)
print("AUC-ROC (classification, higher is better)")
print("=" * 70)
auc_pivot = build_pivot(results_df[results_df['task']=='classification'], 'AUC-ROC')
print(auc_pivot.round(4).to_string())

print("\n" + "=" * 70)
print("Training time (seconds)")
print("=" * 70)
train_pivot = build_pivot(results_df, 'train_time_s')
print(train_pivot.round(2).to_string())

print("\n" + "=" * 70)
print("Inference time per sample (milliseconds)")
print("=" * 70)
inf_pivot = build_pivot(results_df, 'inference_time_per_sample_s') * 1000
print(inf_pivot.round(4).to_string())

rmse_pivot.to_csv('results/tables/pivot_rmse.csv')
acc_pivot.to_csv('results/tables/pivot_accuracy.csv')
auc_pivot.to_csv('results/tables/pivot_auc.csv')
train_pivot.to_csv('results/tables/pivot_train_time.csv')
(inf_pivot).to_csv('results/tables/pivot_inference_time_ms.csv')

print("\n✓ All pivot tables saved to results/tables/")

RMSE (regression, lower is better)
model               xRFM  XGBoost  RandomForest
dataset                                        
concrete          5.7016   5.1542        6.0390
energy            0.6234   0.3542        0.5352
bike_sharing     41.7977  44.6793       49.8290
online_shoppers      NaN      NaN           NaN
shuttle              NaN      NaN           NaN

Accuracy (classification, higher is better)
model              xRFM  XGBoost  RandomForest
dataset                                       
concrete            NaN      NaN           NaN
energy              NaN      NaN           NaN
bike_sharing        NaN      NaN           NaN
online_shoppers  0.8909   0.9023        0.9019
shuttle          0.7867   0.9983        0.9987

AUC-ROC (classification, higher is better)
model              xRFM  XGBoost  RandomForest
dataset                                       
concrete            NaN      NaN           NaN
energy              NaN      NaN           NaN
bike_sharing        NaN

In [12]:
import json, os
nb_path = '02_model_training.ipynb'
with open(nb_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

print(f"Total cells: {len(nb['cells'])}\n")

bad_keywords = ['california_housing', 'wine_quality', 'adult_income',
                'bank_marketing', 'covertype', 'covtype',
                'train_xrfm_bank', 'xrfm_model_cal', 'result_cal_xrfm']

for i, cell in enumerate(nb['cells']):
    src = ''.join(cell.get('source', []))
    cell_type = cell.get('cell_type', '?')
    first_line = src.strip().split('\n')[0][:80] if src.strip() else '(empty)'
    flags = [kw for kw in bad_keywords if kw in src]
    flag_str = f"  ⚠️  CONTAINS: {flags}" if flags else ""
    print(f"Cell {i+1:2d} [{cell_type:8s}]: {first_line}{flag_str}")

Total cells: 17

Cell  1 [markdown]: ## Models
Cell  2 [code    ]: import numpy as np
Cell  3 [code    ]: def evaluate_regression(y_true, y_pred):
Cell  4 [code    ]: def train_xrfm(data, dataset_name):
Cell  5 [code    ]: all_results = []
Cell  6 [code    ]: print("xRFM models in cache:", list(xrfm_models.keys()))
Cell  7 [code    ]: def train_xgboost(data, dataset_name):
Cell  8 [code    ]: xgb_models = {}
Cell  9 [code    ]: def train_random_forest(data, dataset_name):
Cell 10 [code    ]: rf_models = {}
Cell 11 [code    ]: results_df = pd.DataFrame(all_results)
Cell 12 [code    ]: def build_pivot(results_df, metric):
Cell 13 [markdown]: Three models (xRFM, XGBoost, Random Forest) were trained on five datasets each, 
Cell 14 [code    ]: import json, os  ⚠️  CONTAINS: ['california_housing', 'wine_quality', 'adult_income', 'bank_marketing', 'covertype', 'covtype', 'train_xrfm_bank', 'xrfm_model_cal', 'result_cal_xrfm']
Cell 15 [code    ]: print(f"Total entries in all_results: {len(all_

In [13]:
print(f"Total entries in all_results: {len(all_results)}")
print(f"xRFM models cached: {list(xrfm_models.keys()) if 'xrfm_models' in dir() else 'xrfm_models not defined'}")
print(f"XGBoost models cached: {list(xgb_models.keys()) if 'xgb_models' in dir() else 'xgb_models not defined'}")
print(f"RF models cached: {list(rf_models.keys()) if 'rf_models' in dir() else 'rf_models not defined'}")
print(f"\nDATASET_FILES: {DATASET_FILES}")
print(f"\ndatasets dict keys: {list(datasets.keys())}")

Total entries in all_results: 15
xRFM models cached: ['concrete', 'energy', 'bike_sharing', 'online_shoppers', 'shuttle']
XGBoost models cached: ['concrete', 'energy', 'bike_sharing', 'online_shoppers', 'shuttle']
RF models cached: ['concrete', 'energy', 'bike_sharing', 'online_shoppers', 'shuttle']

DATASET_FILES: ['concrete', 'energy', 'bike_sharing', 'online_shoppers', 'shuttle']

datasets dict keys: ['concrete', 'energy', 'bike_sharing', 'online_shoppers', 'shuttle']


In [14]:
import traceback
try:
    result, model = train_xrfm(datasets['concrete'], 'concrete')
    print("SUCCESS")
    print(result)
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")
    traceback.print_exc()


Training xRFM on concrete
None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|                                                                            | 0/1 [00:00<?, ?it/s]

Fitting RFM with ntrain: 659, d: 8, and nval: 165
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 0: 0.03366208076477051 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 1: 0.04313325881958008 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 2: 0.03273177146911621 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 3: 0.0476071834564209 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 4: 0.03816938400268555 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 5: 0.0387880802154541 seconds
Resetting adaptive bandwidth
Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 6: 

Building trees:   0%|                                                                            | 0/1 [00:00<?, ?it/s]

Using cheap batch size
Optimal M batch size: 659
Using SVD
Time taken for round 9: 0.02954578399658203 seconds
Resetting adaptive bandwidth
Tree has no split, stopping training
  Training time: 0.37s (min_subset_size=4000)
Using hard routing for tree prediction
Using hard routing for tree prediction
Using hard routing for tree prediction
Using hard routing for tree prediction
  Metrics: RMSE=5.7016
  Inference time/sample: 0.0096 ms
SUCCESS
{'dataset': 'concrete', 'model': 'xRFM', 'task': 'regression', 'RMSE': 5.701559066772461, 'train_time_s': 0.3655828000046313, 'inference_time_per_sample_s': 9.636569679277712e-06, 'n_train': 659, 'n_features': 8}
